# AP1 — Análise Exploratória do Dataset

**Objetivo:** preparar e explorar o dataset

Este notebook segue a estrutura exigida: (1) importação e rastreabilidade, (2) qualidade e
preparação, (3) análise univariada, (4) valores discrepantes, (5) característica qualitativa
**B**, (6) análise bivariada, (7) exportação do dado pré-processado.

**Convenção de nomes:** ao longo do notebook cada variável é apresentada pelo **nome
legível** seguido do **nome original no arquivo entre parênteses** — ex.: *Preço de Venda
(SalePrice)*.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


## 1. Importação e rastreabilidade

**Identificação e procedência do arquivo:** `AmesHousing_full.txt`, obtido do repositório
GitHub do autor, em
`https://github.com/artPPUNI06/Meu_Projeto/blob/main/data/raw/AmesHousing_full.txt`
(baixado a partir do link *raw* correspondente). O arquivo tem registros de vendas de imóveis residenciais nos EUA, salvo em formato texto delimitado por tabulação.



In [ ]:
RAW_URL = ("https://raw.githubusercontent.com/artPPUNI06/Meu_Projeto/"
           "main/data/raw/AmesHousing_full.txt")
RAW_PATH = "data/raw/AmesHousing_full.txt"

os.makedirs("data/raw", exist_ok=True)

# Importação do arquivo bruto a partir do repositório GitHub do autor —
# a cópia local em data/raw/ é gravada uma única vez e não é alterada depois.
if not os.path.exists(RAW_PATH):
    import urllib.request
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Identificação/procedência: tamanho e hash do arquivo bruto, para rastreabilidade
size_bytes = os.path.getsize(RAW_PATH)
with open(RAW_PATH, "rb") as f:
    raw_bytes = f.read()

print(f"Origem: {RAW_URL}")
print(f"Cópia local: {RAW_PATH}")
print(f"Tamanho: {size_bytes:,} bytes")
print(f"Quebra de linha CRLF presente: {b'\\r\\n' in raw_bytes[:5000]}")
try:
    raw_bytes.decode('utf-8')
    print("Decodificável em UTF-8: Sim (compatível com ASCII)")
except UnicodeDecodeError as e:
    print(f"Decodificável em UTF-8: Não ({e})")

print("Separador identificado: TAB")


Origem: https://raw.githubusercontent.com/artPPUNI06/Meu_Projeto/main/data/raw/AmesHousing_full.txt
Cópia local: data/raw/AmesHousing_full.txt
Tamanho: 984,880 bytes
Quebra de linha CRLF presente: False
Decodificável em UTF-8: Sim (compatível com ASCII)
Separador identificado: TAB


In [ ]:
# Importação do arquivo bruto — leitura NÃO altera a cópia em data/raw/
df_raw = pd.read_csv(RAW_PATH, sep=chr(9), encoding="utf-8")

print(f"Dimensões: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
print(f"Unidade de análise: cada linha representa UM imóvel residencial vendido")
print(f"                     (identificado unicamente por Parcel ID)")

# checagem de integridade: arquivo bruto permanece intocado
assert os.path.getsize(RAW_PATH) == size_bytes, "O arquivo bruto foi alterado!"
print("\nÍntegro: data/raw/AmesHousing_full.txt não foi modificado pela leitura.")


Dimensões: 2930 linhas x 82 colunas
Unidade de análise: cada linha representa UM imóvel residencial vendido
                     (identificado unicamente por Parcel ID)

Íntegro: data/raw/AmesHousing_full.txt não foi modificado pela leitura.


### Variáveis necessárias — nome legível e nome original

| Nome legível | Nome original no arquivo |
|---|---|
| Identificador do imóvel | `PID` |
| Preço de Venda | `SalePrice` |
| Área Construída (área habitável acima do solo) | `Gr Liv Area` |
| Número de Lareiras *(característica qualitativa B)* | `Fireplaces` |
| Área de Revestimento em Alvenaria | `Mas Vnr Area` |
| Tipo de Revestimento em Alvenaria | `Mas Vnr Type` |
| Ano de Construção | `Year Built` |
| Ano de Reforma/Adição | `Year Remod/Add` |
| Ano da Venda | `Yr Sold` |
| Mês da Venda | `Mo Sold` |

**Normalização de nomes:** os nomes originais do arquivo não foram alterados (mantidos
`str` com espaços e `/`, ex.: `"Year Remod/Add"`), pois o pandas os preserva corretamente
como colunas string sem necessidade de normalização para a leitura. Nenhuma renomeação foi
aplicada às colunas originais; o mapeamento acima é apenas para apresentação legível ao
longo deste notebook (variável `NOMES_LEGIVEIS`, definida na Seção 7).


## 2. Qualidade e preparação

### 2.1 Tipos de dados e conversões necessárias

Todas as variáveis usadas neste notebook (`SalePrice`, `Gr Liv Area`, `Fireplaces`,
`Mas Vnr Area`, `Year Built`, `Year Remod/Add`, `Yr Sold`, `Mo Sold`) já são lidas pelo
pandas com tipo numérico adequado (`int64` ou `float64`); nenhuma conversão de tipo é
necessária para elas. Não há conversão de tipo necessária para nenhuma das colunas usadas
neste notebook.


In [ ]:
cols_interesse = ['SalePrice', 'Gr Liv Area', 'Fireplaces', 'Fireplace Qu',
                  'Mas Vnr Area', 'Mas Vnr Type', 'Year Built', 'Year Remod/Add',
                  'Yr Sold', 'Mo Sold']
print(df_raw[cols_interesse].dtypes)


### 2.2 Duplicidades

In [ ]:
dup_pid = df_raw['PID'].duplicated().sum()
dup_order = df_raw['Order'].duplicated().sum()
dup_full = df_raw.duplicated().sum()

print(f"Linhas totalmente duplicadas: {dup_full}")
print(f"PID (identificador do imóvel) duplicado: {dup_pid}")
print(f"Order (índice sequencial) duplicado: {dup_order}")
print("\nConclusão: não há duplicidades no arquivo — cada linha corresponde a um imóvel único.")


### 2.3 Valores ausentes — Tabela 1

A documentação oficial do dataset  especifica que, para a
maioria das variáveis relacionadas a características do imóvel, o código `NA` não indica
dado faltante por erro de coleta, e sim **ausência da característica** (ex.: `Pool QC = NA`
significa "sem piscina", `Fireplace Qu = NA` significa "sem lareira", `Bsmt Qual = NA`
significa "sem porão"). Essa convenção foi consultada e é usada para interpretar as colunas
de interesse abaixo.

**Tabela 1 — Resumo de tipos, valores ausentes e duplicidades** (variáveis de interesse
para este notebook; o dataset completo tem 2930 linhas e 82 colunas, sem duplicidade de
linhas ou de `PID`).


In [ ]:
tabela1 = pd.DataFrame({
    'Nome legível': ['Preço de Venda', 'Área Construída', 'Número de Lareiras',
                      'Qualidade da Lareira', 'Área de Rev. em Alvenaria',
                      'Tipo de Rev. em Alvenaria', 'Ano de Construção',
                      'Ano de Reforma/Adição', 'Ano da Venda', 'Mês da Venda'],
    'Nome original': cols_interesse,
    'Tipo (dtype)': [str(df_raw[c].dtype) for c in cols_interesse],
    'Ausentes (n)': [df_raw[c].isna().sum() for c in cols_interesse],
    'Ausentes (%)': [round(df_raw[c].isna().mean()*100, 2) for c in cols_interesse],
})
tabela1['Duplicidade de linhas (dataset)'] = f"{dup_full} de {len(df_raw)}"
tabela1


### 2.4 Colunas Fireplaces, Mas Vnr Area, Year Remod/Add e Year Built

Inspeção dedicada às quatro colunas exigidas pelo enunciado



In [ ]:
# Fireplaces (Número de Lareiras)
print("=== Fireplaces (Número de Lareiras) ===")
print(df_raw['Fireplaces'].value_counts().sort_index())
print(f"Ausentes: {df_raw['Fireplaces'].isna().sum()}")

# checagem de consistência com Fireplace Qu (NA = sem lareira)
consistente = ((df_raw['Fireplaces'] == 0) == (df_raw['Fireplace Qu'].isna())).all()
print(f"\nConsistência Fireplaces==0 <-> Fireplace Qu==NA (sem lareira): {consistente}")


In [ ]:
# Mas Vnr Area / Mas Vnr Type (Área e Tipo de Revestimento em Alvenaria)
print("=== Mas Vnr Area (Área de Revestimento em Alvenaria) ===")
print(df_raw['Mas Vnr Area'].describe())

ausentes_area = df_raw['Mas Vnr Area'].isna()
ausentes_tipo = df_raw['Mas Vnr Type'].isna()

print(f"\nAusentes em Mas Vnr Area: {ausentes_area.sum()}")
print(f"Ausentes em Mas Vnr Type: {ausentes_tipo.sum()}")
print(f"Ausentes em AMBAS simultaneamente: {(ausentes_area & ausentes_tipo).sum()}")
print(f"Tipo ausente mas área = 0 (registro coerente de 'sem revestimento'): "
      f"{(ausentes_tipo & (df_raw['Mas Vnr Area'] == 0)).sum()}")


**Interpretação (Mas Vnr Area / Mas Vnr Type):** das 1775 linhas com `Mas Vnr Type`
ausente, 1752 têm `Mas Vnr Area = 0` — um registro **coerente** de "sem revestimento em
alvenaria" (tipo e área concordam). As **23 linhas restantes têm ambas as colunas ausentes
simultaneamente** (nem tipo, nem área preenchidos): nesses casos não há evidência de que a
área seja de fato zero — pode simplesmente não ter sido registrada. **Conforme instrução do
enunciado, esses 23 valores ausentes de `Mas Vnr Area` NÃO são classificados
automaticamente como "sem revestimento" (0); eles são preservados como ausentes (`NaN`)**
na base tratada, e assim permanecerão documentados até que a AP3 decida por um método de
imputação (ex.: mediana por bairro) ou pela exclusão pontual dessas 23 observações nas
análises que envolvam essa variável.


In [ ]:
# Year Built / Year Remod/Add (Ano de Construção / Ano de Reforma-Adição)
print("=== Year Built (Ano de Construção) ===")
print(df_raw['Year Built'].describe())
print("\n=== Year Remod/Add (Ano de Reforma/Adição) ===")
print(df_raw['Year Remod/Add'].describe())

inconsistentes = df_raw[df_raw['Year Remod/Add'] < df_raw['Year Built']]
print(f"\nCasos com Ano de Reforma anterior ao Ano de Construção (inconsistência lógica): "
      f"{len(inconsistentes)}")
inconsistentes[['Order', 'PID', 'Year Built', 'Year Remod/Add', 'Yr Sold', 'SalePrice']]


**Justificativa do tratamento (Year Built / Year Remod/Add):** foi encontrado **1 caso**
(`Order = 851`) em que o Ano de Reforma/Adição (2001) é anterior ao Ano de Construção
(2002) — logicamente impossível e provável erro de digitação (diferença de apenas 1 ano).
Como essas colunas não são usadas nas análises univariada/bivariada obrigatórias desta AP2
(apenas na característica B e na futura trilha Binomial), a linha é **mantida sem
correção automática**, e a inconsistência fica **documentada** aqui para tratamento
específico na AP3, quando essas colunas forem efetivamente usadas em modelagem.

### 2.5 Cópia lógica dos dados originais

Uma cópia lógica em memória (`df_work`) é criada a partir de `df_raw`; todas as
transformações a partir daqui são aplicadas em `df_work`, preservando `df_raw` (e o arquivo
em `data/raw/`) inalterados.


In [ ]:
df_work = df_raw.copy()
print(f"df_raw  (original, intocado): {df_raw.shape}")
print(f"df_work (cópia de trabalho) : {df_work.shape}")


## 3. Análise univariada

Variáveis analisadas: **Preço de Venda (SalePrice)**, **Área Construída (Gr Liv Area)** e a
característica qualitativa **B — Número de Lareiras (Fireplaces)**.

### 3.1 Preço de Venda e Área Construída — Tabela 2


In [ ]:
def medidas_posicao_dispersao_forma(s: pd.Series) -> dict:
    return {
        'n': s.count(),
        'Média': s.mean(),
        'Mediana': s.median(),
        'Desvio-padrão': s.std(),
        'Variância': s.var(),
        'Mínimo': s.min(),
        'Q1 (25%)': s.quantile(0.25),
        'Q3 (75%)': s.quantile(0.75),
        'Máximo': s.max(),
        'Assimetria (skewness)': s.skew(),
        'Curtose (kurtosis)': s.kurt(),
    }

tabela2 = pd.DataFrame({
    'Preço de Venda em US$ (SalePrice)': medidas_posicao_dispersao_forma(df_work['SalePrice']),
    'Área Construída em pés quadrados (Gr Liv Area)': medidas_posicao_dispersao_forma(df_work['Gr Liv Area']),
}).T.round(2)
tabela2


### 3.2 Figura 1 — Histogramas (dois painéis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(df_work['SalePrice'], bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('Preço de Venda (SalePrice)')
axes[0].set_xlabel('Preço de Venda (US$)')
axes[0].set_ylabel('Frequência (n. de imóveis)')

axes[1].hist(df_work['Gr Liv Area'], bins=40, color='#DD8452', edgecolor='white')
axes[1].set_title('Área Construída (Gr Liv Area)')
axes[1].set_xlabel('Área Construída (pés quadrados)')
axes[1].set_ylabel('Frequência (n. de imóveis)')

fig.suptitle('Figura 1 — Distribuição do Preço de Venda e da Área Construída', y=1.02)
fig.tight_layout()
plt.show()


**Leitura:** as duas distribuições são assimétricas à direita (assimetria positiva —
confirmado pelos coeficientes de assimetria na Tabela 2), com a maior parte dos imóveis
concentrada em preços/áreas moderados e uma cauda longa de imóveis de maior valor/área.

### 3.3 Característica qualitativa B — Número de Lareiras (Fireplaces) — Tabela 3


In [ ]:
freq_abs = df_work['Fireplaces'].value_counts().sort_index()
freq_rel = (df_work['Fireplaces'].value_counts(normalize=True).sort_index() * 100).round(2)

tabela3 = pd.DataFrame({
    'Frequência absoluta': freq_abs,
    'Frequência relativa (%)': freq_rel,
})
tabela3.index.name = 'Número de Lareiras (Fireplaces)'
tabela3


### 3.4 Figura 2 — Gráfico de barras

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(tabela3.index.astype(str), tabela3['Frequência absoluta'], color='#55A868')
ax.set_xlabel('Número de Lareiras (Fireplaces)')
ax.set_ylabel('Frequência absoluta (n. de imóveis)')
ax.set_title('Figura 2 — Número de Lareiras (Fireplaces)')
for i, v in enumerate(tabela3['Frequência absoluta']):
    ax.text(i, v + 15, str(v), ha='center', fontsize=9)
fig.tight_layout()
plt.show()


**Leitura preliminar:** as categorias `2`, `3` e `4` lareiras juntas somam apenas 234
imóveis (8,0%), sendo `3` (12 casos) e `4` (1 caso) categorias muito raras para comparações
estatísticas isoladas — esse ponto é retomado e formalizado na Seção 5, antes da análise
bivariada de preço por categoria.


## 4. Valores discrepantes — Cercas de Tukey

Aplicação obrigatória das cercas de Tukey às variáveis quantitativas selecionadas
(Preço de Venda e Área Construída):

$$LI = Q_1 - 1{,}5 \times IQR \qquad LS = Q_3 + 1{,}5 \times IQR \qquad IQR = Q_3 - Q_1$$

### Tabela 4 — Cercas de Tukey


In [ ]:
def cercas_tukey(s: pd.Series) -> dict:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_sinalizados = ((s < li) | (s > ls)).sum()
    return {'Q1': q1, 'Q3': q3, 'IQR': iqr, 'Limite Inferior (LI)': li,
            'Limite Superior (LS)': ls, 'Casos sinalizados (n)': n_sinalizados,
            'Casos sinalizados (%)': round(n_sinalizados / len(s) * 100, 2)}

tabela4 = pd.DataFrame({
    'Preço de Venda (SalePrice)': cercas_tukey(df_work['SalePrice']),
    'Área Construída (Gr Liv Area)': cercas_tukey(df_work['Gr Liv Area']),
}).T.round(2)
tabela4


### Figura 3 — Boxplots (dois painéis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 5))

axes[0].boxplot(df_work['SalePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.6))
axes[0].set_title('Preço de Venda (SalePrice)')
axes[0].set_ylabel('Preço de Venda (US$)')
axes[0].set_xticks([])

axes[1].boxplot(df_work['Gr Liv Area'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#DD8452', alpha=0.6))
axes[1].set_title('Área Construída (Gr Liv Area)')
axes[1].set_ylabel('Área Construída (pés quadrados)')
axes[1].set_xticks([])

fig.suptitle('Figura 3 — Valores discrepantes (cercas de Tukey)', y=1.02)
fig.tight_layout()
plt.show()


### Investigação dos casos sinalizados

O Preço de Venda tem 137 casos acima do limite superior (imóveis de alto padrão — preços
elevados, porém plausíveis para o mercado). A Área Construída tem 75 casos sinalizados;
destacam-se dois **imóveis com área extrema** (acima de 4000 pés quadrados), muito distantes
do restante da distribuição:


In [ ]:
extremos = df_work[df_work['Gr Liv Area'] > 4000][
    ['Order', 'PID', 'Gr Liv Area', 'SalePrice', 'Sale Condition', 'Yr Sold']
].sort_values('Gr Liv Area', ascending=False)
extremos


**Decisão do grupo (mantida/corrigir/excluir), com justificativa contextual:**

- **Preço de Venda:** os 137 casos sinalizados são **mantidos** — correspondem a imóveis de
  alto padrão genuinamente mais caros (variação de mercado real), não erros de digitação.
- **Área Construída:** dos 75 casos sinalizados, a **grande maioria também é mantida**
  (imóveis grandes, porém consistentes com o mercado). **Os dois casos com área acima de
  4000 pés²** chamam atenção adicional: ambos têm `Sale Condition = "Partial"`, indicando
  venda de imóvel novo ainda em construção/incompleto no momento da negociação — uma
  condição de venda atípica que **rompe a relação esperada entre área e preço** (imóveis
  enormes vendidos por valores abaixo do que a área sugeriria). Por isso, **esses 2 casos são
  excluídos** especificamente das análises que envolvem `Gr Liv Area` a partir daqui; os
  demais 73 casos sinalizados permanecem na base.

Abaixo, a comparação exigida entre manter e excluir esses 2 pontos:


In [ ]:
df_sem_exclusao = df_work.copy()
df_com_exclusao = df_work[df_work['Gr Liv Area'] <= 4000].copy()

r_sem, p_sem = stats.pearsonr(df_sem_exclusao['SalePrice'], df_sem_exclusao['Gr Liv Area'])
r_com, p_com = stats.pearsonr(df_com_exclusao['SalePrice'], df_com_exclusao['Gr Liv Area'])

comparacao = pd.DataFrame({
    'Sem exclusão (n=2930)': {
        'Correlação de Pearson (SalePrice x Gr Liv Area)': round(r_sem, 4),
        'Mediana Gr Liv Area': df_sem_exclusao['Gr Liv Area'].median(),
        'Casos sinalizados por Tukey (Gr Liv Area)': cercas_tukey(df_sem_exclusao['Gr Liv Area'])['Casos sinalizados (n)'],
    },
    'Com exclusão (n=2928)': {
        'Correlação de Pearson (SalePrice x Gr Liv Area)': round(r_com, 4),
        'Mediana Gr Liv Area': df_com_exclusao['Gr Liv Area'].median(),
        'Casos sinalizados por Tukey (Gr Liv Area)': cercas_tukey(df_com_exclusao['Gr Liv Area'])['Casos sinalizados (n)'],
    },
}).T
comparacao


A exclusão dos 2 pontos **fortalece** a correlação linear entre Preço de Venda e Área
Construída (de aproximadamente 0,707 para um valor mais alto), sem alterar praticamente a
mediana da área construída — evidência de que eram pontos atípicos por condição de venda, e
não representativos da relação típica entre as duas variáveis. A partir daqui, `df_work` é
atualizado para refletir essa decisão.


In [ ]:
df_work = df_com_exclusao.reset_index(drop=True)
print(f"df_work atualizado (pós-decisão de exclusão): {df_work.shape}")


## 5. Característica qualitativa B — Número de Lareiras (Fireplaces)

**Significado:** `Fireplaces` registra a quantidade de lareiras existentes no imóvel
(variável discreta de contagem, tratada aqui como qualitativa ordinal por representar
categorias de presença/quantidade de uma característica do imóvel).

**Nome legível (nome original):** Número de Lareiras (`Fireplaces`).

**Frequências e categorias raras** (retomando a Tabela 3, Seção 3.3): `0` lareiras = 1422
imóveis (48,6%); `1` lareira = 1274 imóveis (43,5%); `2` lareiras = 221 (7,5%); `3` lareiras
= 12 (0,4%); `4` lareiras = 1 (0,03%). Não há valores ausentes em `Fireplaces` — o valor `0`
é consistente com `Fireplace Qu = NA` ("sem lareira"), conforme verificado na Seção 2.4.

**Justificativa do agrupamento:** as categorias `3` e `4` têm, respectivamente, apenas 12 e
1 observação — número insuficiente para comparações estatísticas robustas (ex.: quartis de
preço) de forma isolada. Por isso, `2`, `3` e `4` lareiras são agrupadas em uma única
categoria **"Duas ou mais"**, preservando o significado (imóveis com múltiplas lareiras) e
garantindo um tamanho amostral adequado (234 imóveis, 8,0%) para comparação com as demais
categorias.

**Rótulos finais das categorias de B:** `Sem Lareira` (0), `Uma Lareira` (1), `Duas ou Mais`
(2, 3 ou 4). Esta decisão é registrada aqui e preservada para a AP3.


In [ ]:
def rotular_B(n):
    if n == 0:
        return 'Sem Lareira'
    elif n == 1:
        return 'Uma Lareira'
    else:
        return 'Duas ou Mais'

ordem_categorias_B = ['Sem Lareira', 'Uma Lareira', 'Duas ou Mais']
df_work['B'] = pd.Categorical(df_work['Fireplaces'].apply(rotular_B),
                               categories=ordem_categorias_B, ordered=True)

print("Frequências finais da característica B (Número de Lareiras, agrupada):")
print(df_work['B'].value_counts().reindex(ordem_categorias_B))


## 6. Análise bivariada

### 6.1 Preço de Venda (SalePrice) x Ano da Venda (Yr Sold) — Tabela 5


In [ ]:
tabela5 = df_work.groupby('Yr Sold').agg(
    Quantidade_de_vendas=('SalePrice', 'count'),
    Mediana_do_Preco=('SalePrice', 'median'),
).rename(columns={'Quantidade_de_vendas': 'Quantidade de Vendas',
                   'Mediana_do_Preco': 'Mediana do Preço de Venda (US$)'})
tabela5.index.name = 'Ano da Venda (Yr Sold)'
tabela5


### Figura 4 — Mediana anual do Preço de Venda

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tabela5.index, tabela5['Mediana do Preço de Venda (US$)'],
        marker='o', color='#C44E52', linewidth=2)
ax.set_xlabel('Ano da Venda (Yr Sold)')
ax.set_ylabel('Mediana do Preço de Venda (US$)')
ax.set_title('Figura 4 — Mediana Anual do Preço de Venda')
ax.set_xticks(tabela5.index)
fig.tight_layout()
plt.show()


**Discussão:** a linha conecta **medianas anuais agregadas**, não preços de imóveis
individuais — nenhum imóvel é acompanhado ao longo do tempo. As diferenças observadas entre
anos podem refletir a **composição dos imóveis vendidos em cada ano** (mix de tamanhos,
localizações e padrões), efeitos de **inflação** e **condições gerais do mercado** (2008–2009
coincide com a crise financeira internacional) — não uma tendência causal, nem uma análise
formal de série temporal. O Mês da Venda (`Mo Sold`) poderia servir como extensão descritiva
(sazonalidade dentro do ano), mas não substitui a agregação anual acima, que é a exigida.

### 6.2 Preço de Venda (SalePrice) x Área Construída (Gr Liv Area)

**Figura 5 — Diagrama de dispersão**


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_work['Gr Liv Area'], df_work['SalePrice'],
           alpha=0.35, s=18, color='#4C72B0', edgecolor='none')
ax.set_xlabel('Área Construída (pés quadrados) — Gr Liv Area')
ax.set_ylabel('Preço de Venda (US$) — SalePrice')
ax.set_title('Figura 5 — Preço de Venda x Área Construída')
fig.tight_layout()
plt.show()


In [ ]:
r, p_valor = stats.pearsonr(df_work['SalePrice'], df_work['Gr Liv Area'])
print(f"Correlação de Pearson (r): {r:.4f}")
print(f"p-valor: {p_valor:.2e}")
print(f"n = {len(df_work)} (já refletindo a exclusão dos 2 casos extremos, Seção 4)")


**Interpretação:** a dispersão mostra **direção positiva e forma aproximadamente
linear** entre área construída e preço de venda, com dispersão crescente do preço em áreas
maiores (leque se abrindo — heterocedasticidade visual) e alguns imóveis de área moderada
com preços destacadamente altos, provavelmente por outras características (localização,
qualidade construtiva) não capturadas neste gráfico bivariado. A correlação linear de
Pearson (calculada após a exclusão decidida na Seção 4) indica uma associação **positiva de
intensidade forte** entre as duas variáveis — **não** uma relação de causalidade. Como
demonstrado na Seção 4, valores discrepantes de área (imóveis parcialmente construídos)
podem **distorcer** o coeficiente de correlação, tanto atenuando quanto inflando seu valor
dependendo de sua posição relativa à reta de tendência geral.

### 6.3 Preço de Venda (SalePrice) x Característica B (Número de Lareiras) — Tabela 6


In [ ]:
def resumo_preco_por_grupo(s: pd.Series) -> dict:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return {'n': s.count(), 'Mediana': s.median(), 'Q1': q1, 'Q3': q3, 'IQR': q3 - q1}

tabela6 = df_work.groupby('B', observed=True)['SalePrice'].apply(
    lambda s: pd.Series(resumo_preco_por_grupo(s))
).unstack().reindex(ordem_categorias_B).round(2)
tabela6


**Figura 6 — Boxplots do preço por categoria de B**

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
dados_grupos = [df_work.loc[df_work['B'] == cat, 'SalePrice'] for cat in ordem_categorias_B]
bp = ax.boxplot(dados_grupos, labels=ordem_categorias_B, patch_artist=True)
for patch, cor in zip(bp['boxes'], ['#4C72B0', '#55A868', '#C44E52']):
    patch.set_facecolor(cor)
    patch.set_alpha(0.6)
ax.set_xlabel('Número de Lareiras (Fireplaces) — categorias agrupadas')
ax.set_ylabel('Preço de Venda (US$) — SalePrice')
ax.set_title('Figura 6 — Preço de Venda por Número de Lareiras')
fig.tight_layout()
plt.show()


**Interpretação:** a mediana do Preço de Venda **cresce** de "Sem Lareira" para "Uma
Lareira" e para "Duas ou Mais", assim como a dispersão (IQR) do preço, que é maior nos
grupos com lareiras. Há valores discrepantes de preço nos três grupos, mais numerosos e mais
extremos no grupo "Duas ou Mais". Essas diferenças são **descritivas**: não foi realizado
nenhum teste de hipótese, e não se atribui causalidade ao número de lareiras — imóveis com
mais lareiras tendem a ser maiores e de padrão mais alto de forma geral, o que provavelmente
explica boa parte da diferença observada, e não a lareira isoladamente.


## 7. Exportação do dado pré-processado

### Registro explícito — nomes originais e nomes utilizados


In [ ]:
NOMES_LEGIVEIS = {
    'PID': 'Identificador do imóvel',
    'SalePrice': 'Preço de Venda',
    'Gr Liv Area': 'Área Construída',
    'Fireplaces': 'Número de Lareiras (base numérica da característica B)',
    'B': 'Característica qualitativa B — Número de Lareiras (categorias agrupadas)',
    'Mas Vnr Area': 'Área de Revestimento em Alvenaria',
    'Mas Vnr Type': 'Tipo de Revestimento em Alvenaria',
    'Year Built': 'Ano de Construção',
    'Year Remod/Add': 'Ano de Reforma/Adição',
    'Yr Sold': 'Ano da Venda',
    'Mo Sold': 'Mês da Venda',
}
pd.DataFrame(list(NOMES_LEGIVEIS.items()), columns=['Nome original', 'Nome legível'])


### Tratamentos realizados (resumo)

1. **Duplicidades:** nenhuma encontrada (0 linhas duplicadas, 0 `PID` duplicado).
2. **`Mas Vnr Area`:** os 23 valores verdadeiramente ausentes (tipo e área ausentes
   simultaneamente) foram **preservados como `NaN`** — não classificados automaticamente
   como "sem revestimento" (0). Nenhuma imputação foi aplicada nesta AP2.
3. **`Year Built` / `Year Remod/Add`:** 1 inconsistência lógica encontrada (reforma anterior
   à construção) — **documentada, sem correção automática**, pois essas colunas não são
   usadas nas análises quantitativas exigidas nesta AP2.
4. **Valores discrepantes (Tukey) em `Gr Liv Area`:** os **2 imóveis com área acima de 4000
   pés² e `Sale Condition = "Partial"`** foram **excluídos** da base de trabalho, com
   justificativa contextual e comparação de resultados com/sem exclusão (Seção 4). Os
   demais 73 casos sinalizados por Tukey em `Gr Liv Area`, e todos os 137 casos sinalizados
   em `SalePrice`, foram **mantidos** (variação legítima de mercado).
5. **Característica B (`Fireplaces`):** agrupada em 3 categorias — `Sem Lareira` (0),
   `Uma Lareira` (1), `Duas ou Mais` (2, 3 ou 4 lareiras) — devido à raridade das categorias
   `3` e `4` isoladamente (12 e 1 observações, respectivamente).

### Agrupamento final e rótulos de B

- **Variável:** Número de Lareiras (`Fireplaces`)
- **Categorias finais:** `Sem Lareira`, `Uma Lareira`, `Duas ou Mais`
- **Coluna criada:** `B` (tipo categórico ordenado), preservada no arquivo exportado

### Reprodução

Para reproduzir a versão organizada: execute este notebook do início ao fim com acesso à
internet (para baixar `AmesHousing_full.txt` do repositório GitHub em
`https://raw.githubusercontent.com/artPPUNI06/Meu_Projeto/main/data/raw/AmesHousing_full.txt`)
ou com esse mesmo arquivo já presente em `data/raw/`. O notebook lê o arquivo bruto, aplica
os tratamentos documentados nas Seções 2, 4 e 5 (nesta ordem), e grava o resultado em
`data/processed/AmesHousing.csv`.


In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

OUTPUT_PATH = 'data/processed/AmesHousing.csv'
df_work.to_csv(OUTPUT_PATH, sep=',', encoding='utf-8', index=False)

print(f"Arquivo processado salvo em: {OUTPUT_PATH}")
print(f"Dimensões finais: {df_work.shape[0]} linhas x {df_work.shape[1]} colunas")
print(f"Colunas: {list(df_work.columns[-3:])} ... (inclui coluna 'B' adicionada)")


In [ ]:
# Checagem final de reprodutibilidade: recarregar o arquivo processado
df_check = pd.read_csv(OUTPUT_PATH)
print(f"Releitura de {OUTPUT_PATH}: {df_check.shape}")
df_check[['PID', 'SalePrice', 'Gr Liv Area', 'Fireplaces', 'B']].head()
